# 🤖 NLP Engineer 2 — Gloss → Text Transformer (Seq2Seq)
### Ishara Arabic Sign Language Translator | Graduation Project

**مسؤوليتك:**  
أنت بتبني وتدرّب الـ Transformer Seq2Seq model اللي بيحوّل  
الـ Arabic Gloss sequence لـ Arabic natural language sentence.

---
**الـ Pipeline بتاعتك:**
```
Gloss tokens:  [ هو  معلم  لغه  اشاره ]  →  IDs: [1, 6, 9, 14, 17, 2, 0, ...]
                        ↓
            Transformer Encoder  (Gloss → context)
                        ↓
            Transformer Decoder  (Teacher Forcing during training)
                        ↓
            Cross-Entropy Loss  +  Label Smoothing
                        ↓
Predicted:  [ هو  مدرس  لغه  اشاره ]
```

> **dependency:** محتاج `vocab_gloss.json`, `vocab_text.json`, `IsharaNLP_Dataset` من **NLP Engineer 1**


## 📦 Step 1 — Install & Import

In [ ]:
import os, json, math, numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")
print(f"PyTorch: {torch.__version__}")


## ⚙️ Step 2 — Config

In [ ]:
# ====================================================
CSV_PATH      = "/kaggle/input/.../train.csv"
VOCAB_GLOSS   = "/kaggle/working/vocab_gloss.json"
VOCAB_TEXT    = "/kaggle/working/vocab_text.json"
CKPT_DIR      = "/kaggle/working/checkpoints/"
os.makedirs(CKPT_DIR, exist_ok=True)

MAX_GLOSS_LEN = 12
MAX_TEXT_LEN  = 12
PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

# Transformer hyperparams
D_MODEL    = 256    # embedding dimension
N_HEADS    = 8      # attention heads  (D_MODEL يلزم يكون قابل للقسمة على N_HEADS)
N_LAYERS   = 3      # encoder + decoder layers
D_FF       = 512    # feedforward dimension
DROPOUT    = 0.1
BATCH_SIZE = 64
LR         = 1e-4
EPOCHS     = 50
# ====================================================

# Load vocabs
gloss_vocab = json.load(open(VOCAB_GLOSS, encoding="utf-8"))
text_vocab  = json.load(open(VOCAB_TEXT,  encoding="utf-8"))
idx2text    = {int(v): k for k, v in text_vocab.items()}

GLOSS_VOCAB_SIZE = len(gloss_vocab)
TEXT_VOCAB_SIZE  = len(text_vocab)
print(f"✅ Gloss vocab: {GLOSS_VOCAB_SIZE} | Text vocab: {TEXT_VOCAB_SIZE}")


## 🗄️ Step 3 — Dataset (copy من NLP Eng 1)

انسخ الـ `IsharaNLP_Dataset` و `normalize_arabic` و `tokenize` من **NLP Engineer 1** هنا.


In [ ]:
import re
from collections import Counter

def normalize_arabic(text):
    pass

def tokenize(text, vocab, max_len, normalize=True):
    pass

def detokenize(ids, idx2vocab):
    pass

class IsharaNLP_Dataset(Dataset):
    def __init__(self, df, gloss_vocab, text_vocab, max_gloss=12, max_text=12):
        pass
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        pass

df    = pd.read_csv(CSV_PATH)
n_val = int(len(df) * 0.1)
train_ds = IsharaNLP_Dataset(df.iloc[n_val:], gloss_vocab, text_vocab)
val_ds   = IsharaNLP_Dataset(df.iloc[:n_val], gloss_vocab, text_vocab)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"✅ Train: {len(train_ds)}  |  Val: {len(val_ds)}")


## 🏗️ Step 4 — Positional Encoding

الـ Transformer مش عنده إحساس بترتيب الـ tokens تلقائياً، فمحتاجين نضيف positional encoding.


In [ ]:
class PositionalEncoding(nn.Module):
    """
    Classic sinusoidal positional encoding (Vaswani et al., 2017).
    Adds position information to token embeddings.
    """
    def __init__(self, d_model, max_len=50, dropout=0.1):
        pass
    
    def forward(self, x):
        pass

print("✅ PositionalEncoding defined")


## 🏗️ Step 5 — Transformer Seq2Seq Model

**ليه Transformer وليه مش LSTM؟**  
- الـ self-attention بتشوف كل الـ tokens في نفس الوقت → مش sequential
- بتقدر تشتغل بالـ parallel على الـ GPU → أسرع في الـ training
- بتمسك long-range dependencies أحسن (token 1 يأثر على token 10)


In [ ]:
class GlossToTextTransformer(nn.Module):
    """
    Transformer Seq2Seq for Gloss → Arabic Text translation.
    
    Encoder: receives Gloss token IDs
    Decoder: generates Text token IDs (teacher forcing during training)
    """
    def __init__(self, src_vocab_size, tgt_vocab_size,
                 d_model, n_heads, n_layers, d_ff, dropout, max_len=50):
        pass
    
    def _init_weights(self):
        pass
    
    def make_causal_mask(self, seq_len):
        """Upper-triangular mask to prevent decoder from seeing future tokens."""
        pass
    
    def forward(self, src, tgt):
        """
        src: (batch, src_len)  — Gloss IDs
        tgt: (batch, tgt_len)  — Text IDs (teacher forcing: shifted right)
        
        Returns logits: (batch, tgt_len, tgt_vocab_size)
        """
        pass


# ---- Instantiate + test ----
model = GlossToTextTransformer(
    src_vocab_size=GLOSS_VOCAB_SIZE,
    tgt_vocab_size=TEXT_VOCAB_SIZE,
    d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, d_ff=D_FF, dropout=DROPOUT
).to(device)

dummy_src = torch.randint(0, GLOSS_VOCAB_SIZE, (4, MAX_GLOSS_LEN)).to(device)
dummy_tgt = torch.randint(0, TEXT_VOCAB_SIZE,  (4, MAX_TEXT_LEN)).to(device)
out = model(dummy_src, dummy_tgt)
print(f"✅ Model output shape: {out.shape}")   # (4, 12, TEXT_VOCAB_SIZE)
print(f"✅ Total params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 🧑‍🏫 Step 6 — Teacher Forcing (ازاي بيشتغل؟)

```
Target sentence: <BOS> هو مدرس لغه <EOS>

Decoder INPUT  (shifted right):  <BOS> هو  مدرس  لغه
Decoder OUTPUT (shifted left):         هو  مدرس  لغه <EOS>

في كل خطوة، الـ decoder بياخد الـ token الصح من الـ ground truth
ومش بياخد الـ prediction بتاعته (ده teacher forcing)
```


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    pass


@torch.no_grad()
def evaluate(model, loader, criterion):
    pass

print("✅ train_one_epoch + evaluate defined")


In [ ]:
def run_training(model, train_loader, val_loader, epochs=EPOCHS):
    pass

# history = run_training(model, train_loader, val_loader)
print("✅ run_training defined")


## 🔭 Step 7 — Beam Search Decoding

أثناء الـ inference (مش training)، بنستخدم **Beam Search** بدل greedy  
لأنه بيجرب أكثر من احتمال ويختار الجملة الأحسن كلها.


In [ ]:
@torch.no_grad()
def beam_search(model, src_ids, idx2text, beam_width=4, max_len=12):
    """
    Beam search decoding for a single Gloss input.
    
    Args:
        model      : trained GlossToTextTransformer
        src_ids    : tensor (1, src_len)  — Gloss token IDs
        idx2text   : dict {int: str}
        beam_width : number of candidates to keep at each step
        max_len    : maximum output length
    
    Returns:
        best sentence as string
    """
    pass


# ---- Test ----
# g_ids, t_ids = val_ds[0]
# pred = beam_search(model, g_ids.unsqueeze(0), idx2text, beam_width=4)
# true = detokenize(t_ids.tolist(), idx2text)
# print(f"Predicted: {pred}")
# print(f"True:      {true}")
print("✅ beam_search defined")


## 📐 Step 8 — BLEU Score Evaluation

In [ ]:
def compute_bleu(model, val_loader, idx2text, beam_width=4, max_samples=200):
    """Compute corpus BLEU-1 to BLEU-4 on validation set."""
    pass

# bleu = compute_bleu(model, val_loader, idx2text)
print("✅ compute_bleu defined")


## 📋 Summary — ملخص مسؤوليتك

| Task | Status |
|---|---|
| Copy `IsharaNLP_Dataset` من NLP Eng 1 | ⬜ |
| Implement `PositionalEncoding` | ⬜ |
| Implement `GlossToTextTransformer` | ⬜ |
| Training loop مع teacher forcing | ⬜ |
| Save `stage2_best.pt` checkpoint | ⬜ |
| Implement Beam Search decoding | ⬜ |
| Compute BLEU-1/2/4 on validation set | ⬜ |
| Handoff: `stage2_best.pt` للـ Backend Engineer | ⬜ |

**Target metrics:**
- BLEU-4 ≥ 20 على الـ validation set
- Val Loss ≤ 1.5

**الـ output المطلوب:**
```
checkpoints/
    stage2_best.pt   ← ابعته للـ Backend Engineer
```
> ✉️ بعد ما تخلص، بعّت `stage2_best.pt` للـ **Backend Engineer**.
